# AlgoChowk Quantitative Research & Event-Driven Backtesting Challenge
### Investigating Post-Crash Mean Reversion in NIFTY 50 (2007–2026)
**Candidate**: Dhruv | **Language**: Python 3.13 | **Dataset**: NIFTY 50 Index (`^NSEI`)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_validator import DataValidator
from src.event_detector import EventDetector
from src.statistical_engine import StatisticalEngine
from src.backtester import EventBacktester

sns.set_theme(style="whitegrid")
plt.rcParams.update({'font.sans-serif': 'Arial', 'font.size': 11})
print("Libraries loaded successfully.")

## 1. Data Loading and Validation
We load 19 years of daily NIFTY data (2007–2026) and audit for missing dates, duplicates, and OHLC invariants.

In [ ]:
data_path = os.path.join("..", "data", "nifty50_daily.csv")
raw_df = pd.read_csv(data_path)
validator = DataValidator(raw_df)
clean_df, val_report = validator.validate()
print(f"Validated {val_report['final_rows']} trading days ({val_report['date_range'][0]} to {val_report['date_range'][1]})")
print(f"Duplicates removed: {val_report['duplicate_dates_found']}, Invariant violations: {val_report['ohlc_invariant_violations']}")

## 2. Event Detection & Forward Returns
We scan for one-day falls ($R_t \le -2.0\%$) and compute forward returns across 1, 2, 3, 5, and 10-day holding horizons.

In [ ]:
detector = EventDetector(clean_df)
stat_engine = StatisticalEngine(seed=42)
holding_periods = [1, 2, 3, 5, 10]

baselines = detector.get_unconditional_baseline(holding_periods)
events = detector.detect_events(threshold_pct=-2.0, holding_periods=holding_periods)
print(f"Total drop events found: {len(events)}")
events[['Event_Date', 'Event_Drop_Pct', 'Event_Close', 'Next_Open', 'Fwd_Ret_Close_5d', 'Fwd_Ret_Open_5d']].head(10)

## 3. Statistical Hypothesis Testing vs Unconditional Baseline
We compare post-drop forward returns against unconditional baseline returns using Welch's t-test and a 10,000-sample Circular Block Bootstrap.

In [ ]:
summary_rows = []
for h in holding_periods:
    ev_close = events[f"Fwd_Ret_Close_{h}d"]
    ev_open = events[f"Fwd_Ret_Open_{h}d"]
    base = baselines[h]
    
    comp = stat_engine.compare_against_baseline(ev_close, base)
    boot = stat_engine.run_bootstrap_test(ev_close, base, num_simulations=10000)
    
    summary_rows.append({
        "Horizon": f"{h}d",
        "Baseline Mean (%)": f"{comp['baseline_mean']:+.2f}%",
        "Model A Close Mean (%)": f"{comp['event_mean']:+.2f}%",
        "Abnormal Return (%)": f"{comp['abnormal_mean']:+.2f}%",
        "Welch t-stat": f"{comp['welch_t_stat']:+.2f}",
        "Welch p-value": f"{comp['welch_p_value']:.4f}",
        "Bootstrap p-val": f"{boot['bootstrap_p_value']:.4f}",
        "Significant (p<0.05)": comp['is_statistically_significant_5pct'],
        "Model B Open Mean (%)": f"{ev_open.mean():+.2f}%"
    })

results_df = pd.DataFrame(summary_rows)
results_df

## 4. Visualizations: Forward Returns, Regime Falsification & Equity Curves

In [ ]:
horizons = ["1d", "2d", "3d", "5d", "10d"]
base_means = [baselines[h].mean() for h in holding_periods]
close_means = [events[f"Fwd_Ret_Close_{h}d"].mean() for h in holding_periods]
open_means = [events[f"Fwd_Ret_Open_{h}d"].mean() for h in holding_periods]

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(horizons))
width = 0.25

ax.bar(x - width, base_means, width, label="Unconditional Baseline", color="#7f8c8d")
ax.bar(x, close_means, width, label="Post-Drop (T Close Entry)", color="#2980b9")
ax.bar(x + width, open_means, width, label="Post-Drop (T+1 Open Entry)", color="#e74c3c")

ax.set_xlabel("Forward Holding Horizon")
ax.set_ylabel("Mean Return (%)")
ax.set_title("NIFTY 50 Forward Returns: Event vs Baseline (2007–2026)")
ax.set_xticks(x)
ax.set_xticklabels(horizons)
ax.legend()
plt.show()

## 5. Regime Decomposition: 200 SMA Bull vs Bear Markets

In [ ]:
bull_events = events[events["Regime_Bull"] == True]
bear_events = events[events["Regime_Bull"] == False]

fig, ax = plt.subplots(figsize=(9, 5))
bull_means = [bull_events[f"Fwd_Ret_Close_{h}d"].mean() for h in holding_periods]
bear_means = [bear_events[f"Fwd_Ret_Close_{h}d"].mean() for h in holding_periods]

ax.plot(horizons, bull_means, marker="o", linewidth=2.5, color="#27ae60", label="Bull Regime (Price > 200 SMA, N=54)")
ax.plot(horizons, bear_means, marker="s", linewidth=2.5, color="#c0392b", label="Bear Regime (Price < 200 SMA, N=146)")
ax.plot(horizons, base_means, linestyle="--", color="#7f8c8d", label="Unconditional Baseline")

ax.set_xlabel("Holding Horizon")
ax.set_ylabel("Mean Return (%)")
ax.set_title("Regime Falsification: Bull Dip-Buying vs Bear Falling Knife")
ax.legend()
plt.show()

## 6. Event-Driven Backtest Simulation with Real Frictions

In [ ]:
backtester = EventBacktester(clean_df, initial_capital=1_000_000.0, slippage_bps_per_leg=2.0, statutory_cost_pct=0.03)
bt_res = backtester.run_backtest(events, holding_period_days=5, execution_model="close")

print(f"Total Trades: {bt_res['total_trades']}")
print(f"Total Net Return: {bt_res['total_net_return_pct']:+.2f}%")
print(f"CAGR: {bt_res['cagr_pct']:.2f}%")
print(f"Max Drawdown: {bt_res['max_drawdown_pct']:.2f}%")
print(f"Win Rate: {bt_res['win_rate_pct']:.1f}%")
print(f"Sharpe Ratio: {bt_res['sharpe_ratio']:.2f}")
print(f"Profit Factor: {bt_res['profit_factor']}")